# 🗣️ Babel — Colab Session Template
## Phase 0: Compute & Deployment Environment

**Purpose:** Reusable session-starter for every Babel Colab work session.  
Run this notebook top-to-bottom at the start of every new session to restore the full environment.

---
### ⚡ Before running this notebook:
1. **Runtime → Change runtime type → T4 GPU** (or better)
2. Confirm you are signed into the Google account that owns your Drive folder
3. This notebook does NOT install model weights — that happens in phase1_setup.ipynb

---
### ⚠️ Compute context:
> This project has **NO local NVIDIA GPU**. All model training and inference runs in this
> Colab session. Colab free tier caps at ~12 hours + disconnects on idle.  
> **Every intermediate output is checkpointed to Google Drive immediately.**

## STEP 1 — GPU Verification
Fail-fast: confirm T4 (or better) is attached before doing anything else.

In [ ]:
# Each cell is self-contained — imports are repeated where needed
import subprocess
import sys

print('=' * 60)
print('BABEL SESSION START — GPU VERIFICATION')
print('=' * 60)

# --- nvidia-smi check ---
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode != 0:
    print('❌ No GPU detected!')
    print('   Fix: Runtime → Change runtime type → T4 GPU')
    sys.exit('GPU required. Session aborted.')

print('✅ nvidia-smi: GPU present')

# Print GPU name line from nvidia-smi
for line in result.stdout.split('\n'):
    if any(gpu in line for gpu in ['Tesla', 'T4', 'A100', 'P100', 'V100', 'L4']):
        print(f'   {line.strip()}')

# --- PyTorch CUDA check ---
try:
    import torch
    print(f'\n   PyTorch version : {torch.__version__}')
    print(f'   CUDA available  : {torch.cuda.is_available()}')
    if torch.cuda.is_available():
        gpu_name = torch.cuda.get_device_name(0)
        total_vram = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f'   GPU name        : {gpu_name}')
        print(f'   Total VRAM      : {total_vram:.1f} GB')
        if total_vram < 12:
            print('   ⚠️  WARNING: <12 GB VRAM. Some models may OOM on simultaneous load.')
            print('       Strategy: load → run → del model → torch.cuda.empty_cache() between stages.')
        else:
            print('   ✅ VRAM sufficient for all Babel pipeline models')
    else:
        print('   ❌ CUDA not available despite GPU being present — PyTorch CUDA mismatch')
        print('      Fix: reinstall PyTorch with the correct CUDA version (see STEP 5)')
except ImportError:
    print('   ⚠️  PyTorch not yet installed — will be installed in STEP 5')

print('\n' + '=' * 60)
print('GPU CHECK COMPLETE')
print('=' * 60)

## STEP 2 — Mount Google Drive
All intermediate outputs (transcripts, cloned audio, dubbed video, eval results) are checkpointed here.  
**Never assume anything survives past the current Colab session unless it's on Drive.**

In [ ]:
import os

# Mount Google Drive — you will be prompted to authorize on first run
from google.colab import drive
drive.mount('/content/drive')

# --- Define Drive checkpoint directory ---
# CHECKPOINT: checkpointing outputs to Google Drive
DRIVE_ROOT        = '/content/drive/MyDrive/babel'
DRIVE_OUTPUTS     = DRIVE_ROOT + '/outputs'
DRIVE_CHECKPOINTS = DRIVE_ROOT + '/checkpoints'
DRIVE_LOGS        = DRIVE_ROOT + '/logs'

for d in [DRIVE_ROOT, DRIVE_OUTPUTS, DRIVE_CHECKPOINTS, DRIVE_LOGS]:
    os.makedirs(d, exist_ok=True)
    print(f'  ✅ Drive path ready: {d}')

print(f'\n📁 All outputs will be checkpointed to: {DRIVE_ROOT}')
print('   This persists across Colab sessions and disconnects.')

## STEP 3 — Clone / Pull GitHub Repository
Treat GitHub as where the code lives. Colab is compute-only.  
Pull latest at the start of every session.

In [ ]:
import os
import subprocess

# ─── UPDATE THIS ─────────────────────────────────────────────────────────────
GITHUB_REPO_URL = 'https://github.com/YOUR_USERNAME/babel.git'  # ← your GitHub URL
REPO_DIR = '/content/babel'
# ─────────────────────────────────────────────────────────────────────────────

if os.path.isdir(REPO_DIR):
    print(f'Repository already cloned at {REPO_DIR}.')
    print('Pulling latest changes from GitHub...')
    result = subprocess.run(['git', 'pull'], cwd=REPO_DIR, capture_output=True, text=True)
    print(result.stdout.strip() or '  (already up to date)')
    if result.returncode != 0:
        print(f'  ⚠️  git pull warning: {result.stderr.strip()}')
else:
    print(f'Cloning {GITHUB_REPO_URL}...')
    result = subprocess.run(
        ['git', 'clone', GITHUB_REPO_URL, REPO_DIR],
        capture_output=True, text=True
    )
    if result.returncode == 0:
        print(f'  ✅ Cloned to {REPO_DIR}')
    else:
        print(f'  ❌ Clone failed: {result.stderr}')
        raise RuntimeError('Failed to clone repo. Check GITHUB_REPO_URL above.')

# Show current commit
commit = subprocess.run(
    ['git', 'log', '--oneline', '-1'],
    cwd=REPO_DIR, capture_output=True, text=True
)
print(f'  HEAD: {commit.stdout.strip()}')

## STEP 4 — Install System Dependencies
ffmpeg is required by Whisper and pydub. Run once per Colab session.

In [ ]:
# FIX: subprocess imported here — each cell is self-contained
import subprocess

print('Installing system dependencies (ffmpeg, cmake, build-essential)...')
import subprocess
import os
os.system('apt-get update -qq')
os.system('apt-get install -y -qq ffmpeg cmake build-essential libsndfile1')

# Verify ffmpeg
result = subprocess.run(['ffmpeg', '-version'], capture_output=True, text=True)
if result.returncode == 0:
    first_line = result.stdout.split('\n')[0]
    print(f'  ✅ ffmpeg: {first_line}')
else:
    print('  ❌ ffmpeg not found — install failed')

## STEP 5 — Install PyTorch (Pinned — Must Come First)
⚠️ **Install PyTorch BEFORE all other packages.** If TTS or other packages install first, they pull
in a newer PyTorch that breaks Wav2Lip's face_alignment dependency.

In [ ]:
# FIX: sys imported here for sys.modules reload
import sys
import subprocess

print('Installing pinned PyTorch 1.12.1 + CUDA 11.3...')
print('(~2-3 minutes on first run)')

subprocess.run(
    [
        sys.executable, '-m', 'pip', 'install', '-q',
        'torch==1.12.1+cu113',
        'torchvision==0.13.1+cu113',
        'torchaudio==0.12.1+cu113',
        '--extra-index-url', 'https://download.pytorch.org/whl/cu113',
    ],
    check=True
)

# Reload torch modules in case a different version was previously imported
for mod in list(sys.modules.keys()):
    if mod.startswith('torch'):
        del sys.modules[mod]

import torch
print(f'\n  ✅ torch=={torch.__version__}')
print(f'     CUDA available: {torch.cuda.is_available()}')

assert '1.12' in torch.__version__, (
    f'Expected torch 1.12.x, got {torch.__version__}. '
    'Check for conflicting packages in the pip output above.'
)

## STEP 6 — Install Babel Requirements
Install from requirements.txt (pinned versions). Then install Babel itself in editable mode.

> ℹ️ Uses `subprocess` instead of `!` shell magic so Python variables can be passed safely.

In [ ]:
# FIX: all imports self-contained; use subprocess instead of ! magic for variable paths
import os
import sys
import subprocess

REPO_DIR = '/content/babel'

print('Installing requirements.txt (pinned versions)...')
print('Note: torch is already installed above; pip will skip those lines.')

# FIX: Use subprocess.run instead of !pip so REPO_DIR variable works correctly
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q',
     '-r', os.path.join(REPO_DIR, 'requirements.txt')],
    check=True
)

# Verify numpy was not silently upgraded (Wav2Lip requires <1.24)
import numpy as np
if np.__version__ != '1.23.5':
    print(f'  ⚠️  numpy was upgraded to {np.__version__} — downgrading...')
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q', 'numpy==1.23.5'],
        check=True
    )
    print('  ✅ numpy downgraded to 1.23.5')
else:
    print(f'  ✅ numpy=={np.__version__} (version intact)')

# Install Babel itself in editable mode so modules are importable
print('\nInstalling Babel in editable mode...')
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '-e', REPO_DIR],
    check=True
)

# Verify the import works
sys.path.insert(0, REPO_DIR)
import importlib
import babel as _babel
print(f'  ✅ babel=={_babel.__version__} importable')

## STEP 7 — Session Environment Variables
Set paths for Drive checkpoints and Wav2Lip (if already cloned from a previous session).

In [ ]:
import os

# ─── Drive paths ───────────────────────────────────────────────────────────
os.environ['BABEL_DRIVE_ROOT']        = '/content/drive/MyDrive/babel'
os.environ['BABEL_DRIVE_OUTPUTS']     = '/content/drive/MyDrive/babel/outputs'
os.environ['BABEL_DRIVE_CHECKPOINTS'] = '/content/drive/MyDrive/babel/checkpoints'

# ─── Wav2Lip paths (set if cloned; otherwise set in phase1_setup.ipynb) ────
WAV2LIP_REPO_PATH       = '/content/Wav2Lip'
WAV2LIP_CHECKPOINT_PATH = '/content/Wav2Lip/checkpoints/wav2lip_gan.pth'

if os.path.isdir(WAV2LIP_REPO_PATH):
    os.environ['WAV2LIP_REPO_PATH']       = WAV2LIP_REPO_PATH
    os.environ['WAV2LIP_CHECKPOINT_PATH'] = WAV2LIP_CHECKPOINT_PATH
    print('  ✅ WAV2LIP_REPO_PATH set: ' + WAV2LIP_REPO_PATH)
    ckpt_exists = os.path.isfile(WAV2LIP_CHECKPOINT_PATH)
    # FIX: avoid nested quotes in f-string — use concatenation instead
    ckpt_icon = '✅' if ckpt_exists else '⚠️ '
    ckpt_msg  = 'found' if ckpt_exists else 'NOT FOUND — run phase1_setup.ipynb STEP 8'
    print(f'  {ckpt_icon} Wav2Lip checkpoint: {ckpt_msg}')
else:
    print('  ℹ️  Wav2Lip not yet cloned in this session.')
    print('     Run phase1_setup.ipynb to clone and download the checkpoint.')

print('\nEnvironment variables set:')
for key in ['BABEL_DRIVE_ROOT', 'BABEL_DRIVE_OUTPUTS', 'BABEL_DRIVE_CHECKPOINTS',
            'WAV2LIP_REPO_PATH', 'WAV2LIP_CHECKPOINT_PATH']:
    val = os.environ.get(key, '(not set)')
    print(f'  {key} = {val}')

## STEP 8 — Full Environment Audit
Print all critical package versions. Paste this output into `notes.md` for every session.

In [ ]:
import importlib
import sys
import platform
from datetime import datetime

CRITICAL_PACKAGES = [
    ('torch',          'torch'),
    ('torchvision',    'torchvision'),
    ('numpy',          'numpy'),
    ('opencv-python',  'cv2'),
    ('librosa',        'librosa'),
    ('face-alignment', 'face_alignment'),
    ('soundfile',      'soundfile'),
    ('pydub',          'pydub'),
    ('transformers',   'transformers'),
    ('sentencepiece',  'sentencepiece'),
    ('TTS',            'TTS'),
    ('Pillow',         'PIL'),
    ('gradio',         'gradio'),
    ('spaces',         'spaces'),
    ('tqdm',           'tqdm'),
]

ts = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
print('=' * 65)
print('BABEL SESSION AUDIT — ' + ts)
print('=' * 65)
print('Python  : ' + sys.version.split()[0])
print('Platform: ' + platform.platform())
print()

all_ok = True
for pkg_name, import_name in CRITICAL_PACKAGES:
    try:
        mod     = importlib.import_module(import_name)
        version = getattr(mod, '__version__', 'installed (no __version__)')
        print(f'  {pkg_name:<25} {version}')
    except ImportError:
        print(f'  {pkg_name:<25} NOT INSTALLED')
        all_ok = False

print()
print('=' * 65)
if all_ok:
    print('All packages present')
else:
    print('Some packages missing — run phase1_setup.ipynb to install them')
print('=' * 65)
print('\n→ Paste this block into notes.md → Phase 1 Log for this session')

## STEP 9 — Session Ready ✅
If all checks above passed, the session is ready for work.  
Continue with the appropriate phase notebook for your task:

| Notebook | Purpose |
|---|---|
| `phase1_setup.ipynb` | Install models, run component tests |
| *(phase2_pipeline.ipynb — Phase 2)* | End-to-end pipeline |
| *(phase3_eval.ipynb — Phase 3)* | Prosody conditioning + evaluation |

---
### 🔁 End-of-session checklist
Before Colab disconnects or session ends:
1. Any new/modified code → push to GitHub: run the cell below or manually `git add -A && git commit && git push`
2. Any outputs worth keeping → confirm they are in `/content/drive/MyDrive/babel/outputs/`
3. Note the session GPU-hour consumption in `notes.md` if on Kaggle

In [ ]:
# FIX: all names used here are imported/defined in this cell
import os
import sys

REPO_DIR = '/content/babel'

try:
    import torch
    gpu_info = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only'
except ImportError:
    gpu_info = 'torch not installed'

print('Babel session is ready.')
print('  Repo : ' + REPO_DIR)
print('  Drive: ' + os.environ.get('BABEL_DRIVE_ROOT', 'NOT SET — re-run STEP 2'))
print('  GPU  : ' + gpu_info)
print()
print('Next: open phase1_setup.ipynb to install and test each model.')

In [ ]:
# Run this at END of session to push any code changes back to GitHub
import subprocess
import sys

REPO_DIR   = '/content/babel'
COMMIT_MSG = 'Session update'  # ← change this before running

def _run(cmd, cwd=REPO_DIR):
    r = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True)
    out = (r.stdout + r.stderr).strip()
    if out:
        print(out)
    return r.returncode

_run(['git', 'add', '-A'])
rc = _run(['git', 'commit', '-m', COMMIT_MSG])
if rc == 0:
    _run(['git', 'push'])
    print('Code pushed to GitHub.')
else:
    print('Nothing to commit (or commit failed — check output above).')